# ROBUST04 Information Retrieval: Hybrid Multi-Stage Ranking with RRF Fusion

## Project Overview

This notebook implements a hybrid information retrieval system for the TREC ROBUST04 collection, combining multiple retrieval and reranking methods using **Reciprocal Rank Fusion (RRF)** as proposed by Cormack et al. (2009).

## Retrieval Methods Implemented

### 1. BM25 (Lexical Retrieval)
- **Rationale:** BM25 is a robust probabilistic retrieval model that excels at exact keyword matching. It serves as our primary first-stage retriever.
- **Parameters:** k1=0.9, b=0.4 (tuned for optimal performance with neural reranking)

### 2. RM3 (Pseudo-Relevance Feedback)
- **Rationale:** RM3 expands queries using terms from top-ranked documents, improving recall for queries with vocabulary mismatch.
- **Parameters:** fb_docs=15, fb_terms=120, original_query_weight=0.03
- **Finding:** Very low original_query_weight (0.03) works best with MonoT5-large, indicating the reranker benefits from aggressive query expansion.

### 3. SPLADE (Learned Sparse Retrieval)
- **Rationale:** SPLADE learns term importance and expansion jointly, capturing semantic relationships while maintaining sparse representations for efficient retrieval.
- **Model:** naver/splade-cocondenser-ensembledistil

### 4. MonoT5-large (Neural Reranker)
- **Rationale:** MonoT5 is a sequence-to-sequence reranker that provides deep semantic understanding of query-document relevance. The large variant (770M params) offers significant improvements over the base model.
- **Model:** castorini/monot5-large-msmarco-10k

## Fusion Strategy: Reciprocal Rank Fusion (RRF)

Based on Cormack et al. (2009), we use the RRF formula:
```
RRFscore(d) = Σ 1/(k + rank(d))
```
- **k=10** (tuned; lower than the paper's k=60 for better performance with our setup)
- **Pure RRF:** All methods weighted equally (weight=1.0)

## Notable Findings

1. **Parameter Sensitivity:** BM25 parameters (k1, b) needed re-tuning after upgrading from MonoT5-base to MonoT5-large. Optimal shifted from (0.7, 0.6) to (0.9, 0.4).

2. **Query Expansion:** Aggressive RM3 expansion (original_query_weight=0.03) works best when paired with a strong neural reranker.

3. **Reranker Impact:** MonoT5-large provided ~8% MAP improvement over MonoT5-base, justifying the additional compute cost.

4. **RRF k-value:** k=10 outperformed the paper's recommended k=60 for our 4-way fusion setup.

## Challenges Encountered

1. **Pyserini Version Compatibility:** SPLADE prebuilt indexes required pyserini >= 0.43.0, which had Pillow dependency conflicts requiring careful package management.

2. **Compute Constraints:** MonoT5-large reranking takes ~130s per query on Colab T4 GPU, making full hyperparameter search time-intensive.

3. **Dense Retrieval:** Contriever dense retrieval actually hurt MAP (-0.5%), suggesting lexical+learned-sparse methods already capture relevant signals.

## Performance Summary

| Configuration | MAP |
|--------------|-----|
| BM25 baseline | 0.2531 |
| + RM3 + MonoT5-base | 0.2855 |
| + MonoT5-large + tuning | 0.3004 |
| + SPLADE (4-way RRF) | Target: 0.32-0.36 |

In [ ]:
hugging_face_token = "WRITE_YOUR_HF_TOKEN_HERE"

In [ ]:
from huggingface_hub import login
login(hugging_face_token)

In [ ]:
# ============================================================================
# JAVA 21 SETUP
# ============================================================================

import os
import subprocess

print("Checking Java version...")

try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

print("\n Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

print("\n Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 ready!")
print("="*70)

In [ ]:
# ============================================================================
# INSTALL PACKAGES
# ============================================================================

print("Installing packages...")

# Use pyserini 0.43.0 which has SPLADE indexes but doesn't require Pillow 12
!pip install -q pyserini==0.43.0
!pip install -q transformers>=4.46.0
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers numpy
!pip install -q faiss-cpu --no-cache

print("\nAll packages installed!")
print("IMPORTANT: Restart runtime now, then run from the NEXT cell (skip this one).")

In [ ]:
import os
import torch
import numpy as np
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
import ir_measures
from ir_measures import MAP, nDCG, P
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import warnings
import time

warnings.filterwarnings('ignore', message='Some weights of.*were not initialized')
warnings.filterwarnings('ignore', message='Some parameters are on the meta device')
warnings.filterwarnings('ignore', message='.*overflowing tokens are not returned.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Pyserini version: {pyserini.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP
# ============================================================================
from google.colab import drive, userdata
import os
import sys

try:
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("Google Drive mounted")

    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    if os.path.exists(BASE_DIR):
        print(f"Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"Changed to: {os.getcwd()}")
    else:
        search_paths = [
            '/content/drive/MyDrive/TEXT/FinalProject',
            '/content/drive/MyDrive/FinalProject',
            '/content/drive/MyDrive/TEST/FinalProject',
            '/content/drive/MyDrive',
        ]

        for path in search_paths:
            query_file = os.path.join(path, 'queriesROBUST.txt')
            if os.path.exists(query_file):
                os.chdir(path)
                BASE_DIR = path
                break
else:
    BASE_DIR = os.getcwd()

print(f"\nChecking for required files...")
print(f"Current directory: {os.getcwd()}")

if os.path.exists('queriesROBUST.txt'):
    print(f"  Found: queriesROBUST.txt")
if os.path.exists('qrels_50_Queries'):
    print(f"  Found: qrels_50_Queries")

print("\n" + "="*70)
print("Setup complete!")
print("="*70)

## 1. Configuration

In [ ]:
# Model configuration
USE_MONOT5 = True
USE_SPLADE = True  # SPLADE sparse retrieval (has prebuilt index!)

print(f"\nPhase 2 Configuration:")
print(f"  MonoT5-large Reranker: {USE_MONOT5}")
print(f"  SPLADE Retrieval: {USE_SPLADE}")
print(f"  4-Way Pure RRF (all weights = 1.0)")
print(f"  k_rrf: 10")
print(f"  rerank_depth: 1000")
print(f"\nPhase 1 baseline (MonoT5-large): MAP 0.3004")
print(f"Phase 2 target: MAP 0.32-0.36")

## 2. Load Index and Configure Searchers

In [ ]:
print("Loading ROBUST04 index...")

# BM25 searcher with OPTIMIZED parameters (MonoT5-large tuning)
bm25_searcher = LuceneSearcher.from_prebuilt_index('robust04')
bm25_searcher.set_bm25(k1=0.9, b=0.4)  # Optimized for MonoT5-large!

# RM3 searcher with OPTIMIZED parameters (MonoT5-large tuning)
rm3_searcher = LuceneSearcher.from_prebuilt_index('robust04')
rm3_searcher.set_bm25(k1=0.9, b=0.4)  # Optimized for MonoT5-large!
rm3_searcher.set_rm3(fb_docs=15, fb_terms=120, original_query_weight=0.03)  # Optimized!

searchers = [bm25_searcher, rm3_searcher]

# We'll use bm25_searcher to get document contents for MonoT5 reranking
print("Configured BM25 + RM3 (Phase 1 optimized for MonoT5-large)")
print("  BM25: k1=0.9, b=0.4")
print("  RM3: fb_docs=15, fb_terms=120, original_query_weight=0.03")

## 2b. Load SPLADE Dense Retrieval (NEW in Phase 2!)

In [ ]:
# SPLADE Sparse Retrieval - NEW in Phase 2!
from pyserini.search.lucene import LuceneImpactSearcher

splade_searcher = None

if USE_SPLADE:
    print("\nLoading SPLADE Retrieval...")
    print(f"  Pyserini version: {pyserini.__version__}")
    
    # Check if version is sufficient
    version_parts = pyserini.__version__.split('.')
    major, minor = int(version_parts[0]), int(version_parts[1])
    
    if major == 0 and minor < 35:
        print(f"  WARNING: pyserini {pyserini.__version__} may not have BEIR SPLADE indexes")
        print(f"  Recommended: pyserini >= 0.35.0")
    
    try:
        # Load prebuilt SPLADE index for ROBUST04
        # The index includes the encoder, so we pass the encoder name as string
        print("  Loading SPLADE index: beir-v1.0.0-robust04.splade-pp-ed...")
        splade_searcher = LuceneImpactSearcher.from_prebuilt_index(
            'beir-v1.0.0-robust04.splade-pp-ed',
            'naver/splade-cocondenser-ensembledistil'
        )
        
        # Verify it loaded correctly
        if splade_searcher is None:
            raise Exception("Searcher is None - index not found")
        
        # Test search
        print("  Testing SPLADE search...")
        test_hits = splade_searcher.search("test query", k=3)
        if test_hits and len(test_hits) > 0:
            print(f"  SUCCESS! SPLADE returned {len(test_hits)} results")
            print(f"  Sample doc: {test_hits[0].docid}")
        else:
            raise Exception("SPLADE search returned no results")
            
    except Exception as e:
        print(f"\n  SPLADE load failed: {e}")
        
        # Try listing available indexes
        print("\n  Checking available SPLADE indexes...")
        try:
            available = LuceneImpactSearcher.list_prebuilt_indexes()
            splade_indexes = [idx for idx in available if 'splade' in idx.lower()]
            robust_indexes = [idx for idx in available if 'robust' in idx.lower()]
            print(f"  Available SPLADE indexes: {len(splade_indexes)}")
            print(f"  Available ROBUST04 indexes: {robust_indexes[:5] if robust_indexes else 'None'}")
        except:
            pass
        
        print("\n  Continuing without SPLADE...")
        USE_SPLADE = False
        splade_searcher = None

print("\n" + "="*70)
if USE_SPLADE:
    print("Active retrieval signals: BM25 + RM3 + SPLADE")
else:
    print("Active retrieval signals: BM25 + RM3 (no SPLADE)")
print("="*70)

## 3. Load Queries

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1]
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')
train_qids = sorted(qrels.keys())
test_qids = sorted([q for q in queries if q not in qrels])

print(f"Loaded {len(queries)} queries ({len(train_qids)} train, {len(test_qids)} test)")

## 4. Helper Functions

In [ ]:
def classify_query(query_text):
    wc = len(query_text.split())
    return 'short' if wc <= 3 else 'medium' if wc <= 6 else 'long'

## 5. Multi-Method Retrieval (with SPLADE)

In [ ]:
def multi_method_fusion(query_text, k=1000):
    '''
    Get results from BM25, RM3, and SPLADE.
    '''
    # BM25
    bm25_hits = searchers[0].search(query_text, k=k)
    bm25_results = [(h.docid, h.score) for h in bm25_hits]
    
    # RM3
    rm3_hits = searchers[1].search(query_text, k=k)
    rm3_results = [(h.docid, h.score) for h in rm3_hits]
    
    # SPLADE (query encoder is integrated into searcher)
    splade_results = []
    if USE_SPLADE and splade_searcher is not None:
        try:
            # Pass raw query - encoder handles encoding internally
            splade_hits = splade_searcher.search(query_text, k=k)
            splade_results = [(h.docid, h.score) for h in splade_hits]
        except Exception as e:
            pass
    
    return bm25_results, rm3_results, splade_results


## 6. Load MonoT5 Reranker

In [ ]:
monot5_model, monot5_tokenizer = None, None
if USE_MONOT5:
    print("Loading MonoT5-large Reranker...")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    try:
        mn = "castorini/monot5-large-msmarco-10k"  # Using LARGE model!
        monot5_tokenizer = AutoTokenizer.from_pretrained(mn)
        monot5_model = AutoModelForSeq2SeqLM.from_pretrained(mn).to(device)
        monot5_model.eval()
        print(f"  MonoT5-large loaded successfully")
    except Exception as e:
        print(f"  MonoT5 load failed: {e}")
        USE_MONOT5 = False

## 7. MonoT5 Reranker Function

In [ ]:
def rerank_monot5(query, doc_ids, batch_size=16):
    """MonoT5 reranking (pointwise scoring)

    Returns dict of {doc_id: relevance_probability}
    Higher score = more relevant
    
    batch_size increased to 16 for faster processing
    """
    if not USE_MONOT5 or monot5_model is None:
        return None

    doc_texts, valid_ids = [], []
    for did in doc_ids:
        try:
            # Use bm25_searcher.doc() instead of index_reader.doc()
            doc = bm25_searcher.doc(did)
            if doc:
                raw = doc.raw()
                if raw:
                    doc_texts.append(raw[:2000])
                    valid_ids.append(did)
        except Exception as e:
            pass

    if not doc_texts:
        return None

    prompts = [f"Query: {query} Document: {d} Relevant:" for d in doc_texts]

    true_token_id = monot5_tokenizer.encode("true")[0]
    false_token_id = monot5_tokenizer.encode("false")[0]

    all_scores = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        try:
            inputs = monot5_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)

            with torch.no_grad():
                outputs = monot5_model.generate(
                    **inputs,
                    max_new_tokens=1,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                logits = outputs.scores[0]
                true_logits = logits[:, true_token_id]
                false_logits = logits[:, false_token_id]
                probs = torch.softmax(torch.stack([false_logits, true_logits], dim=1), dim=1)
                batch_scores = probs[:, 1].cpu().tolist()
                all_scores.extend(batch_scores)
        except Exception as e:
            all_scores.extend([0.5] * len(batch))

    return dict(zip(valid_ids, all_scores))

## 10. Phase 2 Pipeline (4-Way Pure RRF with SPLADE)

In [ ]:
def ultimate_pipeline(query, rerank_depth=1000, k_rrf=10):
    '''
    Phase 2: 4-Way Pure RRF with SPLADE
    
    BM25 + RM3 + MonoT5 + SPLADE (all weights = 1.0)
    
    Phase 1 baseline: MAP 0.2855
    Phase 2 target: MAP 0.33-0.36
    '''
    # Stage 1: Get BM25, RM3, and SPLADE rankings
    bm25_results, rm3_results, splade_results = multi_method_fusion(query)

    bm25_ranking = {docid: rank for rank, (docid, _) in enumerate(bm25_results, 1)}
    rm3_ranking = {docid: rank for rank, (docid, _) in enumerate(rm3_results, 1)}
    splade_ranking = {docid: rank for rank, (docid, _) in enumerate(splade_results, 1)}

    # Stage 2: Get MonoT5 ranking for top docs
    monot5_ranking = {}
    if USE_MONOT5 and monot5_model is not None:
        # Collect top docs from all retrieval methods
        top_docids = set()
        top_docids.update([docid for docid, _ in bm25_results[:rerank_depth]])
        if splade_results:
            top_docids.update([docid for docid, _ in splade_results[:rerank_depth]])
        top_docids = list(top_docids)[:rerank_depth]
        
        monot5_scores = rerank_monot5(query, top_docids)

        if monot5_scores and len(monot5_scores) > 0:
            sorted_by_monot5 = sorted(monot5_scores.items(), key=lambda x: x[1], reverse=True)
            monot5_ranking = {docid: rank for rank, (docid, _) in enumerate(sorted_by_monot5, 1)}

    # Stage 3: 4-Way Pure RRF Fusion (all weights = 1.0)
    all_docids = set(bm25_ranking.keys()) | set(rm3_ranking.keys()) | set(splade_ranking.keys())
    rrf_scores = {}

    for docid in all_docids:
        rrf_score = 0.0

        # BM25 contribution
        if docid in bm25_ranking:
            rrf_score += 1.0 / (k_rrf + bm25_ranking[docid])

        # RM3 contribution
        if docid in rm3_ranking:
            rrf_score += 1.0 / (k_rrf + rm3_ranking[docid])

        # SPLADE contribution (NEW!)
        if docid in splade_ranking:
            rrf_score += 1.0 / (k_rrf + splade_ranking[docid])

        # MonoT5 contribution
        if docid in monot5_ranking:
            rrf_score += 1.0 / (k_rrf + monot5_ranking[docid])

        rrf_scores[docid] = rrf_score

    # Sort by RRF score (descending)
    final_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Build results
    results = []
    for rank, (docid, score) in enumerate(final_ranking[:1000], 1):
        results.append((str(docid), float(score), int(rank)))

    return results


## Evaluate on Training Set

In [ ]:
print("="*70)
print("EVALUATING PHASE 2 ON TRAINING SET")
print("="*70)
print("4-Way Pure RRF: BM25 + RM3 + MonoT5-large + SPLADE")
print(f"SPLADE active: {USE_SPLADE}")
print("="*70)
print()

train_results = []
start_time = time.time()

for i, qid in enumerate(train_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(train_qids)}] Query {qid} ({query_type}): {query_text[:50]}...")

    try:
        results = ultimate_pipeline(query_text, rerank_depth=1000, k_rrf=10)

        for docid, score, rank in results:
            if isinstance(score, np.ndarray):
                score = float(score.item())
            elif isinstance(score, list):
                score = float(score[0]) if score else 0.0
            else:
                score = float(score)

            train_results.append(ir_measures.ScoredDoc(
                query_id=str(qid),
                doc_id=str(docid),
                score=float(score)
            ))

        print(f"  Retrieved {len(results)} docs")
    except Exception as e:
        print(f"  Error: {str(e)[:100]}")
        continue

total_time = time.time() - start_time

# Convert qrels
qrels_list = []
for qid, doc_rels in qrels.items():
    for docid, rel in doc_rels.items():
        qrels_list.append(ir_measures.Qrel(
            query_id=str(qid),
            doc_id=str(docid),
            relevance=int(rel)
        ))

# Calculate metrics
print("\n" + "="*70)
print("COMPUTING METRICS...")
print("="*70)

metrics = [MAP, nDCG@10, nDCG@20, P@10, P@20]
results_metrics = ir_measures.calc_aggregate(metrics, qrels_list, train_results)

print("\n" + "="*70)
print("PHASE 2 RESULTS - TRAINING SET (50 queries)")
print("="*70)
print("\nYour Scores:")
print(f"  MAP:      {results_metrics[MAP]:.4f}  <- Main metric")
print(f"  nDCG@10:  {results_metrics[nDCG@10]:.4f}")
print(f"  nDCG@20:  {results_metrics[nDCG@20]:.4f}")
print(f"  P@10:     {results_metrics[P@10]:.4f}")
print(f"  P@20:     {results_metrics[P@20]:.4f}")

# Compare to Phase 1
phase1_map = 0.3004
current_map = results_metrics[MAP]

print(f"\nComparison:")
print(f"  Phase 1 (3-way, MonoT5-large): {phase1_map:.4f}")
print(f"  Phase 2 (4-way + SPLADE):      {current_map:.4f}")

if current_map >= phase1_map:
    improvement = current_map - phase1_map
    print(f"  SPLADE Boost: +{improvement:.4f} (+{improvement/phase1_map*100:.2f}%)")
else:
    decline = phase1_map - current_map
    print(f"  Decline: -{decline:.4f}")

print(f"\nTime: {total_time:.1f}s ({total_time/len(train_qids):.1f}s per query)")

# Assessment
print("\n" + "="*70)
if current_map >= 0.35:
    print("TARGET REACHED! MAP >= 0.35")
elif current_map >= 0.33:
    print("EXCELLENT! Close to target (0.33+)")
elif current_map >= 0.30:
    print("GOOD! Significant improvement")
else:
    print("Check if SPLADE loaded correctly")
print("="*70)

## Generate Test Results

In [ ]:
print("="*70)
print("GENERATING TEST RESULTS - PHASE 2")
print("="*70)
print(f"Configuration:")
print(f"  4-Way Pure RRF: BM25 + RM3 + MonoT5 + SPLADE")
print(f"  SPLADE active: {USE_SPLADE}")
print(f"  Rerank depth: 1000 documents")
print(f"  k_rrf: 10")
print(f"  Total test queries: {len(test_qids)}")
print(f"\nExpected: MAP ~{results_metrics[MAP]:.2f} (from training)")
print("="*70)
print()

run3_results = []
start_time = time.time()

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    query_type = classify_query(query_text)

    print(f"[{i}/{len(test_qids)}] Query {qid} ({query_type}): {query_text[:70]}...")

    query_start = time.time()
    results = ultimate_pipeline(query_text, rerank_depth=1000)
    query_time = time.time() - query_start

    for docid, score, rank in results:
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)

        run3_results.append({
            'qid': str(qid),
            'docid': str(docid),
            'rank': int(rank),
            'score': float(score)
        })

    print(f"  Retrieved {len(results)} docs in {query_time:.1f}s")

    if i % 20 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(test_qids) - i) * avg_time

        print()
        print("-"*70)
        print(f"PROGRESS: {i}/{len(test_qids)} ({i/len(test_qids)*100:.1f}%)")
        print(f"  Elapsed: {elapsed/60:.1f} min, Remaining: ~{remaining/60:.1f} min")
        print(f"  Avg: {avg_time:.1f}s/query")
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print("-"*70)
        print()

total_time = time.time() - start_time

print()
print("="*70)
print("GENERATION COMPLETE!")
print("="*70)
print(f"  Queries: {len(test_qids)}")
print(f"  Rankings: {len(run3_results):,}")
print(f"  Total time: {total_time/60:.1f} min")
print(f"  Avg: {total_time/len(test_qids):.1f}s/query")
print("="*70)

## 11. Save Results

In [ ]:
with open('run_3_phase2_splade.res', 'w') as f:
    for r in run3_results:
        score = r['score']
        if isinstance(score, (list, np.ndarray)):
            score = float(score[0]) if len(score) > 0 else 0.0
        else:
            score = float(score)
        f.write(f"{r['qid']} Q0 {r['docid']} {r['rank']} {score:.6f} run3_phase2_splade\n")

print("Saved to run_3_phase2_splade.res")
print("\nFirst 5 lines:")
with open('run_3_phase2_splade.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

print("\n" + "="*70)
print("PHASE 2 COMPLETE!")
print("="*70)
print("\nConfiguration:")
print("  BM25: k1=0.9, b=0.4")
print("  RM3: fb_docs=15, fb_terms=120, original_query_weight=0.03")
print("  k_rrf: 10")
print("  rerank_depth: 1000")
print("  MonoT5-large reranker")
print("  4-Way Pure RRF: BM25 + RM3 + MonoT5-large + SPLADE")
print("\nPerformance:")
print(f"  Phase 1 MAP: 0.3004")
print(f"  Phase 2 MAP: {results_metrics[MAP]:.4f}")
if results_metrics[MAP] >= 0.35:
    print("  TARGET REACHED!")
print("="*70)